### 02 EDA: as perguntas.

Uma seção por pergunta, na ordem do enunciado.

As ressalvas metodológicas vem do notebook 1.

In [1]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)

from src.data.load import build_panel, build_transitions, load_config
from src.data.clean import limpar
from src.data.validate import validar_painel, validar_transicoes, resumo_carga

cfg = load_config('../config.yaml')
RAW = '../' + cfg['paths']['raw']

from src.features.build import adicionar_features
from src.analysis.eda_tecnica import *

painel, _log = limpar(build_panel(RAW))
pf = adicionar_features(painel)
trans = build_transitions(pf)
co = coorte_fechada(painel, ['defasagem'])
print(f'{len(painel)} linhas | coorte fechada: {len(co)} alunos')

3030 linhas | coorte fechada: 468 alunos


#### Pergunta 1 — Perfil de defasagem (IAN)

In [2]:
def faixa(d):
    return pd.cut(d, [-99,-3,-1,99], labels=['Severo','Moderado','Adequado'])

sub = painel[painel.ra.isin(co)]
ct = pd.crosstab(sub.ano, faixa(sub.defasagem), normalize='index').round(3)
ct

defasagem,Severo,Moderado,Adequado
ano,,,
2022,0.015,0.658,0.327
2023,0.004,0.577,0.419
2024,0.004,0.344,0.652


In [3]:
from scipy import stats
w = sub.pivot_table(index='ra', columns='ano', values='defasagem')
print(f'Wilcoxon 2022 vs 2024: p = {stats.wilcoxon(w[2022], w[2024]).pvalue:.2g}')
print(f'melhoraram: {(w[2024]>w[2022]).mean():.1%} | pioraram: {(w[2024]<w[2022]).mean():.1%}')

Wilcoxon 2022 vs 2024: p = 7.5e-35
melhoraram: 58.8% | pioraram: 10.3%


**Resposta.** Alunos em nível adequado dobraram: 32,7% → 65,2%. Individualmente, 58,8% melhoraram contra 10,3% que pioraram.

#### Pergunta 2 Desempenho (IDA)

A resposta depende de como se olha, e as duas leituras são verdadeiras.

In [4]:
por_fase = painel.pivot_table(index='fase_num', columns='ano', values='ida', aggfunc='mean').round(2)
por_fase['delta'] = (por_fase[2024] - por_fase[2022]).round(2)
print('Por FASE (base completa):'); display(por_fase)

wi = painel[painel.ra.isin(co)].pivot_table(index='ra', columns='ano', values='ida').dropna()
print('\nMesmos ALUNOS (coorte fechada):', wi.mean().round(2).to_dict())
print(f'Friedman p = {stats.friedmanchisquare(wi[2022], wi[2023], wi[2024]).pvalue:.2g}')

Por FASE (base completa):


ano,2022,2023,2024,delta
fase_num,,,,
0.0,7.14,7.42,7.32,0.18
1.0,6.46,6.81,6.79,0.33
2.0,5.41,6.74,6.25,0.84
3.0,5.14,5.75,5.35,0.21
4.0,6.05,6.00,5.88,-0.17
5.0,5.87,5.90,6.45,0.58
6.0,6.69,6.81,7.23,0.54
7.0,5.25,7.81,5.81,0.56
8.0,NaN,NaN,8.00,NaN



Mesmos ALUNOS (coorte fechada): {2022: 6.68, 2023: 6.77, 2024: 6.21}
Friedman p = 0.0016


### Perguntas 3 e 5 Engajamento e psicossocial antecedem?

Correlação bruta não responde: IEG e IDA compõem o INDE, e metade dos alunos "cai" por regressão à média. Exige controle pelo nível em t, e erro-padrão clusterizado por aluno.

In [5]:
def ols_cluster(y, Xd, g, nomes):
    """OLS com erro-padrão robusto agrupado por aluno."""
    X = np.c_[np.ones(len(Xd)), Xd]
    XtX = np.linalg.pinv(X.T @ X); b = XtX @ X.T @ y; u = y - X @ b
    meat = np.zeros((X.shape[1],) * 2)
    for k in np.unique(g):
        m = g == k; s = X[m].T @ u[m]; meat += np.outer(s, s)
    se = np.sqrt(np.diag(XtX @ meat @ XtX)); t = b / se
    r2 = 1 - (u**2).sum() / ((y - y.mean())**2).sum()
    return pd.DataFrame({'var': ['const'] + nomes, 'coef': b.round(3), 't': t.round(2)}), r2

lag = pf[['ra','ano','ipv_pct','ieg_pct','ida_pct']].copy(); lag['ano'] -= 1
trans = trans.merge(lag.rename(columns={c: c+'_t1' for c in ['ipv_pct','ieg_pct','ida_pct']}),
                    on=['ra','ano'], how='left')

d = trans.dropna(subset=['ida_t1','ida_pct','ieg_pct'])
for nome, cols in [('só IDA_t', ['ida_pct']), ('IDA_t + IEG_t', ['ida_pct','ieg_pct'])]:
    r, r2 = ols_cluster(d.ida_t1.values, d[cols].values, d.ra.values, cols)
    print(f'{nome}: R² = {r2:.4f}'); display(r)

só IDA_t: R² = 0.2665


,var,coef,t
0,const,4.627,43.29
1,ida_pct,3.400,20.68


IDA_t + IEG_t: R² = 0.2801


,var,coef,t
0,const,4.343,37.05
1,ida_pct,3.044,16.20
2,ieg_pct,0.857,4.54


In [6]:
# P5 — IPS antecede queda? (percentil: obrigatório, dado o drift)
for alvo in ['ida_pct_t1','ieg_pct_t1']:
    base = alvo.replace('_t1','')
    d = trans.dropna(subset=[alvo, base, 'ips_pct'])
    r, r2 = ols_cluster(d[alvo].values, d[[base,'ips_pct']].values, d.ra.values, [base,'ips_pct'])
    print(f'{alvo}: R² = {r2:.3f}'); display(r)

ida_pct_t1: R² = 0.289


,var,coef,t
0,const,0.159,9.56
1,ida_pct,0.518,20.91
2,ips_pct,0.108,4.36


ieg_pct_t1: R² = 0.321


,var,coef,t
0,const,0.159,9.35
1,ieg_pct,0.576,25.54
2,ips_pct,0.040,1.67


**P3.** Sim: IEG antecede IDA (t = 4,54) e IPV (t = 7,29), controlando o nível anterior.

**P5.** Parcial: IPS antecede queda de IDA (t = 4,36), mas **não** de IEG (t = 1,67, p = 0,09).

## Pergunta 4 — Autoavaliação

In [7]:
d = pf.dropna(subset=['iaa','ida']).copy()
d['quintil'] = d.groupby('ano').ida.transform(lambda s: pd.qcut(s, 5, labels=False, duplicates='drop'))
res = d.groupby('quintil').agg(n=('iaa','size'), IDA_real=('ida','mean'), IAA_autoavaliado=('iaa','mean')).round(2)
print('correlação Spearman:', round(d.iaa.corr(d.ida, method='spearman'), 3))
res

correlação Spearman: 0.135


,n,IDA_real,IAA_autoavaliado
quintil,,,
0,545,3.54,8.50
1,507,5.51,8.61
2,554,6.71,8.63
3,493,7.70,8.73
4,507,8.85,8.92


Resposta. Não. Alunos com IDA 3,54 se autoavaliam em 8,50; os de IDA 8,85, em 8,92. Correlação de 0,135.

*Quem mais precisa de ajuda não sabe que precisa*

## Pergunta 6 — Psicopedagógico

o IAN é a defasagem recodificada em 10/5/2,5, determinístico em 99,93% das linhas. Não há duas medidas para comparar.

In [8]:
d = trans[trans.ano == 2023].dropna(subset=['ipp','defasagem','fase_ideal_num']).copy()
d['ipp_pct'] = d.ipp.rank(pct=True)
for nome, cols in [('sem IPP', ['defasagem','fase_ideal_num']),
                   ('com IPP', ['defasagem','fase_ideal_num','ipp_pct'])]:
    r, r2 = ols_cluster(d.alvo_piora.values.astype(float), d[cols].values, d.ra.values, cols)
    print(f'{nome}: R² = {r2:.4f}')
print('correlação IPP x defasagem:', round(d.ipp.corr(d.defasagem, method='spearman'), 3))

sem IPP: R² = 0.1334
com IPP: R² = 0.2092
correlação IPP x defasagem: 0.191


## Pergunta 8 — Composição do INDE

O dicionário lista os sete componentes sem informar a ponderação. Recuperados por regressão.

In [9]:
pesos, r2 = recuperar_pesos_inde(painel, 2024)
print(f'R² = {r2:.6f}')
print({k: v for k, v in pesos.items() if k != 'const'})

contribuicao_inde(painel, 2024)

R² = 1.000000
{'iaa': np.float64(0.1), 'ieg': np.float64(0.2), 'ips': np.float64(0.1), 'ipp': np.float64(0.1), 'ida': np.float64(0.2), 'ipv': np.float64(0.2), 'ian': np.float64(0.1)}


,indicador,peso,dp,contrib,pct_variancia
1,ieg,0.2,2.847,0.569,32.001
4,ida,0.2,2.132,0.426,23.961
6,ian,0.1,2.504,0.250,14.074
5,ipv,0.2,1.049,0.210,11.786
2,ips,0.1,1.428,0.143,8.025
0,iaa,0.1,0.909,0.091,5.109
3,ipp,0.1,0.897,0.090,5.043


Resposta. R² = 1,000000 os coeficientes são os pesos, não estimativas: 0,20 para IEG/IDA/IPV e 0,10 para IAA/IPS/IPP/IAN.

A pergunta "o que eleva mais o INDE" é portanto álgebra. O que interessa é peso × dispersão: IEG (32,0%) e IDA (24,0%) respondem por 56% da variação. O IPV tem o mesmo peso nominal mas contribui 11,8%, porque todos estão agrupados perto do mesmo valor.

### Pergunta 10 — Efetividade

recalcular assumindo que todos os alunos que saíram tenham piorado.

In [10]:
for ano, prox in [(2022,2023), (2023,2024)]:
    base = set(painel.loc[painel.ano == ano, 'ra']); segue = set(painel.loc[painel.ano == prox, 'ra'])
    obs = trans[trans.ano == ano]; mel = (obs.delta_defasagem > 0).sum()
    n_sai, n_tot = len(base - segue), len(base)
    print(f'{ano}->{prox}: observado {mel/len(obs):.3f} | '
          f'pior caso {mel/n_tot:.3f} | melhor caso {(mel+n_sai)/n_tot:.3f}')

2022->2023: observado 0.308 | pior caso 0.215 | melhor caso 0.517
2023->2024: observado 0.409 | pior caso 0.309 | melhor caso 0.554


Resposta. A melhora dentro da coorte é inequívoca: 58,8% melhoraram contra 10,3% que pioraram, com Wilcoxon p = 7,5e-35. Já a comparação entre ciclos é mais frágil do que parece. Os limites de Manski por janela são [21,5%; 51,7%] e [30,9%; 55,4%], e como se sobrepõem, os dados sozinhos não provam que o segundo ciclo foi melhor. O que sustenta a comparação é a atrição parecida nos dois anos, 30,2% e 24,6%, o que é suposição e não garantia.


#### Pergunta 11  O que não perguntaram

##### Evasão tem preditores próprios

In [11]:
s24 = set(painel.loc[painel.ano == 2024, 'ra'])
d23 = painel[painel.ano == 2023].copy(); d23['saiu'] = (~d23.ra.isin(s24)).astype(int)
linhas = []
for c in ['ieg','ipv','ida','iaa','ips','defasagem']:
    a, b = d23[d23.saiu == 1][c].dropna(), d23[d23.saiu == 0][c].dropna()
    linhas.append({'indicador': c, 'evadiu': a.mean(), 'permaneceu': b.mean(),
                   'p': stats.mannwhitneyu(a, b).pvalue})
pd.DataFrame(linhas).round(4)

,indicador,evadiu,permaneceu,p
0,ieg,8.2302,8.8648,0.0000
1,ipv,7.7784,8.1157,0.0000
2,ida,6.2025,6.8257,0.0000
3,iaa,8.4698,8.6794,0.0435
4,ips,5.0134,5.1563,0.3096
5,defasagem,-0.8554,-0.5895,0.0001


**O engajamento é o preditor mais forte de evasão** (p = 7e-14) e é praticamente nulo para defasagem (AUC univariado 0,523).

#### Quem o programa realmente beneficia?

In [12]:
w = painel.pivot_table(index='ra', columns='ano', values='defasagem').dropna()
pedra22 = painel[painel.ano == 2022].set_index('ra')[['pedra','defasagem']]
d = w.join(pedra22, how='inner'); d['delta'] = d[2024] - d[2022]
d.groupby(['pedra', d.defasagem.clip(-3, 0)]).delta.agg(['count','mean']).round(2)

count  mean
pedra    defasagem             
Ametista -3.0           6  2.17
         -2.0          40  1.35
         -1.0         105  0.91
          0.0          67 -0.07
Quartzo  -3.0           1  1.00
         -2.0          10  0.30
         -1.0          17  0.47
          0.0           2 -0.50
Topázio  -2.0           7  1.29
         -1.0          29  1.17
          0.0          70  0.44
Ágata    -2.0          26  0.81
         -1.0          74  0.43
          0.0          14 -0.36

Partindo da mesma defasagem inicial (−2), Ametista avança +1,35 e Quartzo apenas +0,30. O programa beneficia mais quem já estava melhor.

Reportado como indício forte, não fato: n de Quartzo é pequeno (30 na corte).